# Synthèse comparative des méthodes

Vue d'ensemble **qualitative** des méthodes décrites dans les trois notebooks précédents
(`01_methodes_statistiques`, `02_apprentissage_automatique`, `03_reseaux_neurones`), selon des
critères pertinents pour la prédiction probabiliste de matchs de la LNH.

Les cotes (élevé / moyen / faible) résument la littérature et les illustrations ; elles ne sont
pas des résultats expérimentaux sur les données de la LNH (voir `03_experimentation/`).

Avec `SAUVER = True`, la figure est enregistrée en PNG (300 dpi) dans
`reports/map6009/figures/` et le tableau en CSV dans `reports/map6009/tables/`
(préfixe `methodes_`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nhl_forecast.config import PROJECT_ROOT
from nhl_forecast.visualization.style import ENCRE, ENCRE_2, appliquer_style

In [ ]:
SAUVER = True  # True : enregistre aussi les figures (PNG) et tableaux (CSV) pour le rapport
FIG_DIR = PROJECT_ROOT / "reports" / "map6009" / "figures"
TAB_DIR = PROJECT_ROOT / "reports" / "map6009" / "tables"

appliquer_style()


def sauver(fig, nom):
    if SAUVER:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"{nom}.png", dpi=300, bbox_inches="tight", facecolor="white")


def sauver_tableau(df, nom):
    if SAUVER:
        TAB_DIR.mkdir(parents=True, exist_ok=True)
        df.to_csv(TAB_DIR / f"{nom}.csv", encoding="utf-8")
    return df

## Critères

| Critère | Question |
|---|---|
| Relations non linéaires | Le modèle capte-t-il interactions et non-linéarités sans ingénierie manuelle ? |
| Dynamique temporelle native | L'évolution de la force (forme, momentum) est-elle modélisée directement ? |
| Incertitude sur les paramètres | Le modèle quantifie-t-il l'incertitude sur la force ou les coefficients ? |
| Calibration native | Les probabilités sont-elles bien calibrées sans recalibration ? |
| Interprétabilité | Peut-on expliquer simplement une prédiction ? |
| Faible besoin en données | Le modèle reste-t-il fiable avec quelques saisons (≈ 1 300 matchs chacune) ? |
| Faible coût de calcul | Entraînement et mise à jour sont-ils rapides ? |

Cotes : 2 = élevé, 1 = moyen, 0 = faible.

In [ ]:
CRITERES = [
    "Relations non linéaires",
    "Dynamique temporelle native",
    "Incertitude sur les paramètres",
    "Calibration native",
    "Interprétabilité",
    "Faible besoin en données",
    "Faible coût de calcul",
]
cotes = pd.DataFrame(
    [
        ("Statistiques", "Régression logistique", 0, 0, 0, 2, 2, 2, 2),
        ("Statistiques", "Elo (et variantes)", 0, 2, 0, 1, 2, 2, 2),
        ("Statistiques", "Glicko / Glicko-2", 0, 2, 2, 1, 2, 2, 2),
        ("Statistiques", "Approches bayésiennes", 1, 1, 2, 2, 1, 1, 0),
        ("Apprentissage automatique", "Forêt aléatoire", 2, 0, 0, 0, 1, 1, 1),
        ("Apprentissage automatique", "XGBoost", 2, 0, 0, 1, 1, 1, 1),
        ("Apprentissage automatique", "LightGBM", 2, 0, 0, 1, 1, 1, 2),
        ("Apprentissage automatique", "SVM", 2, 0, 0, 0, 0, 1, 0),
        ("Réseaux de neurones", "MLP", 2, 0, 0, 1, 0, 0, 1),
        ("Réseaux de neurones", "LSTM / GRU", 2, 2, 0, 1, 0, 0, 0),
        ("Réseaux de neurones", "TCN / attention", 2, 2, 0, 1, 0, 0, 0),
    ],
    columns=["Famille", "Méthode", *CRITERES],
).set_index(["Famille", "Méthode"])
SYMBOLES = {2: "●", 1: "◐", 0: "○"}
sauver_tableau(cotes.replace(SYMBOLES), "methodes_tab_synthese_criteres")
cotes.replace(SYMBOLES)

## Matrice synthèse (figure)

Symboles pleins, demi-pleins et vides : lisibles en noir et blanc. Les lignes horizontales
séparent les trois familles de méthodes.

In [ ]:
n_l, n_c = cotes.shape
fig, ax = plt.subplots(figsize=(8.6, 5.6))
style_marqueur = {2: dict(fillstyle="full"), 1: dict(fillstyle="left"), 0: dict(fillstyle="none")}
for i, (_, ligne) in enumerate(cotes.iterrows()):
    for j, v in enumerate(ligne):
        ax.plot(
            j,
            i,
            marker="o",
            ms=13,
            color=ENCRE,
            mec=ENCRE,
            mew=1.2,
            ls="",
            markerfacecoloralt="white",
            **style_marqueur[int(v)],
        )
familles = cotes.index.get_level_values(0)
for i in range(1, n_l):
    if familles[i] != familles[i - 1]:
        ax.axhline(i - 0.5, color=ENCRE_2, lw=1)
debut = 0
for i in range(1, n_l + 1):
    if i == n_l or familles[i] != familles[i - 1]:
        ax.text(
            n_c - 0.35,
            (debut + i - 1) / 2,
            familles[i - 1].replace(" ", "\n", 1),
            rotation=0,
            ha="left",
            va="center",
            fontsize=8,
            style="italic",
            color=ENCRE_2,
        )
        debut = i
ax.set_xticks(range(n_c), [c.replace(" ", "\n", 1) for c in CRITERES], fontsize=8)
ax.xaxis.tick_top()
ax.set_yticks(range(n_l), cotes.index.get_level_values(1), fontsize=9)
ax.set_xlim(-0.6, n_c - 0.4)
ax.set_ylim(n_l - 0.5, -0.5)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
poignees = [
    plt.Line2D(
        [],
        [],
        marker="o",
        ms=10,
        color=ENCRE,
        ls="",
        markerfacecoloralt="white",
        label=lab,
        **style_marqueur[v],
    )
    for v, lab in [(2, "Élevé"), (1, "Moyen"), (0, "Faible")]
]
ax.legend(
    handles=poignees,
    loc="upper center",
    bbox_to_anchor=(0.45, -0.02),
    ncols=3,
    fontsize=8,
    frameon=False,
)
sauver(fig, "methodes_synthese_matrice")
plt.show()

## Lecture

- Les méthodes **statistiques** (logistique, Elo, Glicko) sont simples, peu gourmandes en
  données et produisent naturellement des probabilités ; Elo et Glicko intègrent la dynamique
  temporelle mais n'exploitent qu'une dimension de force.
- Les méthodes **bayésiennes** se distinguent par la quantification de l'incertitude et la
  régularisation par l'a priori, au prix du coût de calcul.
- Les **ensembles d'arbres** (forêt, XGBoost, LightGBM) exploitent les relations non
  linéaires entre les statistiques, mais la dynamique doit être encodée dans les variables
  (forme récente, momentum) et la calibration doit être vérifiée.
- Les **réseaux de neurones**, surtout temporels, sont les plus flexibles mais aussi les plus
  exigeants en données, un enjeu important avec ≈ 1 300 matchs par saison.

Aucune famille ne domine sur tous les critères : d'où l'intérêt d'une comparaison
expérimentale fondée sur des règles de score propres (log loss, score de Brier) et sur la
calibration.